# Fake News Classification — BiLSTM Deep Learning Model

Dataset: [WELFake_Dataset.csv](https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification)

This notebook builds a **Bidirectional LSTM** text classifier from scratch. Compared to a simple LSTM/dense model, this version fixes the usual accuracy killers:

- combines `title` + `text` for more signal
- removes duplicates and noisy characters (URLs, HTML, punctuation, digits, stopwords)
- picks sequence length from the *actual* data distribution instead of an arbitrary number
- uses a **stacked Bidirectional LSTM** (captures context in both directions) with `SpatialDropout1D` + `Dropout` for regularization
- uses **class weights**, **stratified splits**, **EarlyStopping** and **ReduceLROnPlateau**

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import re
import json
import pickle

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

from tqdm import tqdm
tqdm.pandas()

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


TensorFlow version: 2.21.0
GPU available: []


## 2. Load & Explore Data

In [2]:
df = pd.read_csv('/kaggle/input/fake-news-classification/WELFake_Dataset.csv')
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/fake-news-classification/WELFake_Dataset.csv'

In [ ]:
print(df.isnull().sum())
print()
print(df['label'].value_counts())

sns.countplot(x='label', data=df)
plt.title('Label distribution')
plt.show()

**Note on labels:** documentation for this dataset is inconsistent about which value means "fake" vs "real" — some sources say `0 = fake, 1 = real`, others the opposite. Rather than assume, sample a few rows yourself and check the headlines against the label before you report final results (e.g. sensational/all-caps headlines vs. neutral ones). The code below trains on `0`/`1` as-is, which works either way — you only need the mapping right when you *describe* the results.

In [ ]:
df[['title', 'label']].sample(10, random_state=1)

## 3. Clean & Prepare Text

In [ ]:
# Drop the stray index column, fill missing text, combine title + text for more signal
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c], errors='ignore')
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['content'] = (df['title'] + ' ' + df['text']).str.strip()

df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

before = len(df)
df = df[df['content'].str.len() > 0]
df = df.drop_duplicates(subset='content').reset_index(drop=True)
print(f"Dropped {before - len(df)} empty/duplicate rows -> {len(df)} rows remain")

In [ ]:
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)   # URLs
    text = re.sub(r'<.*?>', ' ', text)                     # HTML tags
    text = re.sub(r'\S+@\S+', ' ', text)                   # emails
    text = re.sub(r'[^a-z\s]', ' ', text)                  # keep letters only
    text = re.sub(r'\s+', ' ', text).strip()               # collapse whitespace
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 1]
    return ' '.join(tokens)

# quick sanity check
print(clean_text(df['content'].iloc[0])[:300])

In [ ]:
df['clean_content'] = df['content'].progress_apply(clean_text)
df = df[df['clean_content'].str.len() > 0].reset_index(drop=True)
print(df.shape)

In [ ]:
df['word_count'] = df['clean_content'].apply(lambda x: len(x.split()))
print(df['word_count'].describe())

p95 = int(df['word_count'].quantile(0.95))
print("95th percentile word count:", p95)

plt.figure(figsize=(8, 4))
sns.histplot(df['word_count'], bins=60)
plt.axvline(p95, color='red', linestyle='--', label=f'95th pct = {p95}')
plt.xlim(0, 1200)
plt.title('Cleaned word-count distribution')
plt.legend()
plt.show()

MAX_LEN = min(p95, 500)   # cap so training stays fast
print("Using MAX_LEN =", MAX_LEN)

## 4. Train / Validation / Test Split

A stratified split keeps the real/fake ratio consistent across all three sets.

In [ ]:
X = df['clean_content'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 5. Tokenization & Padding

In [ ]:
VOCAB_SIZE = 30000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

actual_vocab_size = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print("Words seen in training data:", len(tokenizer.word_index))
print("Vocabulary size used by the model:", actual_vocab_size)

## 6. Build the BiLSTM Model

Stacked **Bidirectional LSTM** layers read the sequence forwards and backwards, so the model
picks up context clues (e.g. sarcasm/hedging framed earlier in a sentence) that a single
one-directional LSTM misses. `SpatialDropout1D` and `Dropout` guard against overfitting.

In [ ]:
EMBED_DIM = 128

model = Sequential([
    Embedding(input_dim=actual_vocab_size, output_dim=EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Train the Model

`class_weight` compensates for any imbalance between the two classes. `EarlyStopping` restores
the best-performing weights instead of whatever the last epoch happened to produce, and
`ReduceLROnPlateau` lowers the learning rate automatically if validation loss stalls.

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 8. Evaluate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print()
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 9. Try It on New Text

In [ ]:
def predict_news(text, model=model, tokenizer=tokenizer, max_len=MAX_LEN):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    prob = float(model.predict(pad, verbose=0)[0][0])
    label = int(prob > 0.5)
    return label, prob

sample_text = "Breaking: scientists confirm the earth is flat, NASA admits decades-long cover-up"
label, prob = predict_news(sample_text)
print(f"Predicted class: {label} (probability = {prob:.4f})")

## 10. Save Model & Tokenizer

Saved to `/kaggle/working/` so they show up under the notebook's **Output** panel and can be
downloaded or reused in another notebook via "Add Input" -> this notebook's output.

In [ ]:
model.save('/kaggle/working/fake_news_bilstm_model.keras')

with open('/kaggle/working/tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

with open('/kaggle/working/model_config.json', 'w') as f:
    json.dump({'MAX_LEN': MAX_LEN, 'VOCAB_SIZE': actual_vocab_size}, f)

print("Saved model, tokenizer, and config to /kaggle/working/")

## Going Further

If you want to push accuracy even higher later:

- **Pretrained embeddings**: add the `glove.6B` dataset as an input and initialize the
  `Embedding` layer with GloVe vectors instead of training embeddings from scratch — usually a
  quick win on datasets this size.
- **Transformer/BERT**: fine-tuning a `DistilBERT` (via Hugging Face `transformers`) will
  typically edge out an LSTM, at the cost of longer training time and needing a GPU.
- **Error analysis**: look at the misclassified rows (compare `y_test` vs `y_pred`) to see if
  there's a pattern — e.g. very short articles, or a specific topic the model struggles with.